# RTENSO2026 El Nino cases: North Pacific and North America circulation

Monthly versions of the CPC ENSO-update panels "Atmospheric anomalies over the North Pacific and
North America" (120E-60W, 20N-70N), for each event in `EVENTS` and each start:

* **500-hPa height**: total height (contours, dam) and anomaly (shading, m).
* **200-hPa wind**: total wind speed (shading, m/s) and vectors.
* **925-hPa temperature**: anomaly (shading, K).

One figure per event, start and field: rows are the target months (`TARGET_MONTHS`, Nov-Feb around
the peak), columns the observations (ERA5, through 2019) and each case's ensemble mean. Forecast
anomalies remove the Reanalysis 1981-2010 lead-time climatology (lead-drift corrected for Feb/Aug
starts, `CLIM_METHOD`); forecast totals add those anomalies to the observed (ERA5) 1981-2010 monthly
climatology, so they carry no model drift. The 2026 starts have no observations to compare with,
since ERA5 here ends in 2019.

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import warnings
from datetime import datetime

import cftime
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

from matplotlib.colors import BoundaryNorm, ListedColormap
from esp_lab.utils import colormap_utils as mycolors

import rtenso_case as rc

warnings.filterwarnings("ignore", message="invalid value encountered")

### Run controls

In [ ]:
derivation_mode = "auto"
CLIM_METHOD = "lead_drift"         # Feb/Aug drift correction: "lead_drift" or "target_month"
MONTH_NAMES = {1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun", 7: "Jul", 8: "Aug",
               9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"}

# Events: rows are TARGET_MONTHS of each event; "init_months" are the starts (one figure each).
EVENTS = {
    2015: {"label": "2015/16 El Niño", "init_months": (2, 5, 8, 11)},
    2026: {"label": "2026/27 El Niño", "init_months": (2, 5, 8)},
}
TARGET_MONTHS = [(0, 11), (0, 12), (1, 1), (1, 2)]   # (years after the event year, month): Nov-Feb

# Panels: Z500 (height + anomaly), WIND200 (200-hPa wind), T925 (925-hPa temperature anomaly)
PANELS = ["Z500", "WIND200", "T925"]
DOMAIN = (120.0, 300.0, 20.0, 70.0)   # lon0, lon1 (0-360), lat0, lat1

# Cases (columns after the observations). Each case reads <post_root>/<case_prefix>_<year><MM>0100/ENxx/post/...
MODEL_PREFIX = "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5"
CASES = {
    "Reanalysis-CLIM": {"case_prefix": f"{MODEL_PREFIX}_RTENSO2026_clim", "init_months": (2, 5, 8, 11)},
    "Reanalysis-OBS": {"case_prefix": f"{MODEL_PREFIX}_RTENSO2026_obs", "init_months": (2, 5, 8, 11)},
    "Reanalysis-NSIDC": {"case_prefix": f"{MODEL_PREFIX}_RTENSO2026_obs_ice_nsidc", "init_months": (2, 5, 8, 11)},
    "Reanalysis-FOSIRL": {"case_prefix": f"{MODEL_PREFIX}_BruteForce", "init_months": (2, 5, 8, 11)},
}
CASES = {name: case for name, case in CASES.items() if case["init_months"]}   # an empty tuple drops a case
FIELDS = ["Z500", "U200", "V200", "T925"]   # loaded for the panels above

: 

### Load observations and forecasts

In [ ]:
%%time
def target_times(year):
    return [cftime.DatetimeNoLeap(year + dy, m, 15) for dy, m in TARGET_MONTHS]


def subset(da):
    lon0, lon1, lat0, lat1 = DOMAIN
    return da.sel(lon=slice(lon0 - 5, lon1 + 5), lat=slice(lat0 - 5, lat1 + 5))


def at_targets(da, times):
    """Forecast (lead, ...) or observation (time, ...) values at the target months, along "time"."""
    if "lead" in da.dims:
        da = da.swap_dims(lead="time").drop_vars("lead")
    return da.reindex(time=times)


# Observed 1981-2010 monthly climatology (for totals) and anomalies
obs_clim = {f: subset(rc.obs_monthly_climatology(f, mode=derivation_mode)).load() for f in FIELDS}
obs_anom = {}
for year in EVENTS:
    for f in FIELDS:
        try:
            obs_anom[(year, f)] = at_targets(subset(rc.obs_anomaly(f, years=(year, year + 2), mode=derivation_mode)),
                                             target_times(year)).load()
        except ValueError as err:   # no ERA5 after 2019
            print(f"[{year}] no {f} observations: {err}")

# Forecast ensemble-mean anomalies and leads at the target months
fc_anom, fc_lead = {}, {}
for year, ev in EVENTS.items():
    times = target_times(year)
    for init in ev["init_months"]:
        for case in CASES:
            if init not in CASES[case]["init_months"]:
                continue
            for f in FIELDS:
                anom = rc.hindcast_anomaly(CASES[case]["case_prefix"], init, f, year=year, method=CLIM_METHOD, mode=derivation_mode)
                if anom is None:
                    break
                fc_anom[(year, case, init, f)] = at_targets(subset(anom.mean("member")), times).load()
                fc_lead[(year, init)] = [int(l) for l in at_targets(anom["lead"].assign_coords(time=anom["time"]), times).values]
            else:
                print(f"[{year} {case} init{init:02d}] loaded")
                continue
            print(f"[{year} {case} init{init:02d}] not available")

### North Pacific and North America maps

In [ ]:
# ---- Figure setup ----
fontz = 24        # base font size; every other size scales from it
fig_type = "png"  # saved figure type: "png", "pdf" (vector), "eps", "svg", ...
fig_dpi = 300     # resolution of saved raster figures
FIG = {
    "panel": (8.5, 3.2),            # inches per map panel (width, height)
    "hspace": 0.35,
    "wspace": 0.08,
    "top": 0.9,
    "suptitle_y": 0.97,
    "suptitle": fontz * 1.0,
    "title": fontz * 0.7,
    "label": fontz * 0.8,
    "tick": fontz * 0.6,
    "coast": {"color": "limegreen", "lw": 1.2},   # CPC-style green coastlines
    "lon_ticks": [150, 180, 210, 240, 270, 300],
    "lat_ticks": [20, 30, 40, 50, 60, 70],
}
STYLE = {
    # anomaly shading (interval, range) and contour intervals
    "Z500": {"ci": 15.0, "range": 150.0, "anom_line": 30.0, "total_line": 60.0, "units": "m"},
    "T925": {"ci": 0.5, "range": 5.0, "units": "K"},
    "WIND200": {"levels": [10, 20, 30, 40, 50, 60, 70], "line": 10.0, "stride": 5, "quiver_scale": 900.0, "units": "m/s",
                "colors": ["#e3e1f6", "#b9b3ee", "#7d6fd8", "#fff6a8", "#f3b33f", "#e2621f", "#9f1d14"]},
}
PANEL_LETTERS = "abcdefghijklmnopqrstuvwxyz"
TITLES = {"Z500": "500-hPa height (contours, dam) and anomaly (shading)",
          "WIND200": "200-hPa wind speed (shading) and vectors",
          "T925": "925-hPa temperature anomaly"}


def lon_label(lon):
    lon = lon % 360
    return "180°" if lon == 180 else (f"{lon:g}°E" if lon < 180 else f"{360 - lon:g}°W")


def new_map(fig, nrow, ncol, index, title):
    ax = fig.add_subplot(nrow, ncol, index, projection=ccrs.PlateCarree(central_longitude=180))
    ax.set_extent(list(DOMAIN), crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.COASTLINE, edgecolor=FIG["coast"]["color"], linewidth=FIG["coast"]["lw"], zorder=5)
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, lw=0.5, color="0.5", alpha=0.6, ls=":")
    gl.top_labels = gl.right_labels = False
    gl.xlocator = plt.FixedLocator([(l + 180) % 360 - 180 for l in FIG["lon_ticks"]])
    gl.ylocator = plt.FixedLocator(FIG["lat_ticks"])
    gl.xlabel_style = gl.ylabel_style = {"size": FIG["tick"]}
    ax.set_title(title, fontsize=FIG["title"], loc="left")
    return ax


def empty_panel(ax, text):
    ax.text(0.5, 0.5, text, transform=ax.transAxes, ha="center", va="center", fontsize=FIG["label"], color="0.4",
            bbox={"facecolor": "white", "edgecolor": "0.6", "pad": 6}, zorder=6)


def contour_lines(ax, da, interval, scale=1.0, labels=True, lw=0.9):
    vals = da * scale
    lo, hi = np.floor(float(vals.min()) / interval) * interval, np.ceil(float(vals.max()) / interval) * interval
    cs = ax.contour(da["lon"], da["lat"], vals, levels=np.arange(lo, hi + interval / 2, interval), colors="k",
                    linewidths=lw, transform=ccrs.PlateCarree(), zorder=3)
    if labels and len(cs.levels):
        ax.clabel(cs, cs.levels[::2], fmt="%g", fontsize=FIG["tick"] * 0.8, inline=True)
    return cs


def draw_panel(ax, kind, anom, clim):
    """One map. ``anom``/``clim``: dicts of field -> (lat, lon) anomaly / observed climatology."""
    tr = ccrs.PlateCarree()
    if kind in ("Z500", "T925"):
        s = STYLE[kind]
        levels = np.arange(-s["range"], s["range"] + s["ci"] / 2, s["ci"])
        cmap = mycolors.blue2red_cmap(levels)
        a = anom[kind]
        cf = ax.contourf(a["lon"], a["lat"], a, levels=levels, cmap=cmap, extend="both", transform=tr, zorder=1)
        if kind == "Z500":   # total height in dam, as in the CPC panels
            contour_lines(ax, a + clim["Z500"], s["total_line"], scale=0.1)
        return cf
    s = STYLE["WIND200"]
    u, v = anom["U200"] + clim["U200"], anom["V200"] + clim["V200"]
    speed = np.hypot(u, v)
    cmap = ListedColormap(s["colors"][:-1])
    cmap.set_over(s["colors"][-1])
    cmap.set_under("white")
    cf = ax.contourf(u["lon"], u["lat"], speed, levels=s["levels"], cmap=cmap, extend="both", transform=tr, zorder=1)
    contour_lines(ax, speed, s["line"], lw=0.6)
    k = s["stride"]
    ax.quiver(u["lon"].values[::k], u["lat"].values[::k], u.values[::k, ::k], v.values[::k, ::k], transform=tr,
              scale=s["quiver_scale"], width=0.0018, color="k", zorder=4)
    return cf


def month_clim(field, t):
    return obs_clim[field].sel(month=t.month).drop_vars("month")


for year, ev in EVENTS.items():
    times = target_times(year)
    for init in ev["init_months"]:
        cases = [c for c in CASES if (year, c, init, "Z500") in fc_anom]
        if not cases:
            continue
        nrow, ncol = len(times), 1 + len(cases)
        leads = fc_lead[(year, init)]
        for kind in PANELS:
            needed = ["U200", "V200"] if kind == "WIND200" else [kind]
            fig = plt.figure(figsize=(FIG["panel"][0] * ncol, FIG["panel"][1] * nrow))
            fig.subplots_adjust(hspace=FIG["hspace"], wspace=FIG["wspace"], top=FIG["top"])
            cf, k = None, 0
            for r, t in enumerate(times):
                label = f"{MONTH_NAMES[t.month]} {t.year}"
                clim = {f: month_clim(f, t) for f in FIELDS}
                ax = new_map(fig, nrow, ncol, r * ncol + 1, f"({PANEL_LETTERS[k]}) Observed (ERA5), {label}")
                k += 1
                obs = {f: obs_anom[(year, f)].sel(time=t) for f in needed if (year, f) in obs_anom}
                if len(obs) == len(needed) and all(bool(o.notnull().any()) for o in obs.values()):
                    cf = draw_panel(ax, kind, obs, clim)
                else:
                    empty_panel(ax, "No observations")
                for c, case in enumerate(cases, start=1):
                    ax = new_map(fig, nrow, ncol, r * ncol + c + 1, f"({PANEL_LETTERS[k]}) {case}, {label} (lead {leads[r]})")
                    k += 1
                    cf = draw_panel(ax, kind, {f: fc_anom[(year, case, init, f)].sel(time=t) for f in needed}, clim)
            cbar = fig.colorbar(cf, ax=fig.axes, orientation="vertical", fraction=0.012, pad=0.015, shrink=0.8, aspect=40)
            cbar.set_label(f"{TITLES[kind].split(' (')[0] if kind != 'Z500' else '500-hPa height anomaly'} ({STYLE[kind]['units']})",
                           fontsize=FIG["label"])
            cbar.ax.tick_params(labelsize=FIG["tick"])
            fig.suptitle(f"{TITLES[kind]}: 1 {MONTH_NAMES[init]} {year} start, {ev['label']}",
                         fontsize=FIG["suptitle"], y=FIG["suptitle_y"])
            print(rc.save_figure(fig, "rtenso2026", "npac", kind, f"y{year}", f"init{init:02d}", dpi=fig_dpi, ext=fig_type))
            plt.show()